In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import os

In [2]:
cols_fecha = ['year', 'month', 'day']
cols_elo = ['elo_h', 'elo_a']
# Añadimos las nuevas métricas: % de victorias y las diferencias matemáticas (Deltas)
cols_avanzadas = ['rest_days_home', 'rest_days_away', 'avg_pts_scored_home', 
                  'avg_pts_allowed_home', 'avg_pts_scored_away', 'avg_pts_allowed_away',
                  'win_pct_5_home', 'win_pct_5_away', 'elo_diff', 'rest_diff']
cols_equipos = []

In [3]:
# 1. One-Hot Encoding de fechas y equipos
def preparar_datos_ohe(df, cols_equipos):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy['year'] = df_copy['game_date'].dt.year
    df_copy['month'] = df_copy['game_date'].dt.month
    df_copy['day'] = df_copy['game_date'].dt.day

    df_copy = pd.get_dummies(df_copy, columns=['team_abbreviation_home','team_abbreviation_away'], dtype=float)
    
    if cols_equipos == []:
        cols_equipos = [c for c in df_copy.columns if 'team_abbreviation_home_' in c or 'team_abbreviation_away_' in c]
        
    return df_copy, cols_equipos

In [4]:
# 2. MEJORA v1.3: Features Avanzadas + Rachas + Deltas
def generar_features_avanzadas(df):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy = df_copy.sort_values('game_date').reset_index(drop=True)

    # Preparar df de locales y visitantes para el historial
    home_df = df_copy[['game_date', 'team_id_home', 'pts_home', 'pts_away']].rename(
        columns={'team_id_home': 'team_id', 'pts_home': 'pts_scored', 'pts_away': 'pts_allowed'})
    home_df['is_home'] = 1

    away_df = df_copy[['game_date', 'team_id_away', 'pts_away', 'pts_home']].rename(
        columns={'team_id_away': 'team_id', 'pts_away': 'pts_scored', 'pts_home': 'pts_allowed'})
    away_df['is_home'] = 0

    team_games = pd.concat([home_df, away_df]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    # Columna auxiliar para saber si ganaron el partido
    team_games['won_game'] = (team_games['pts_scored'] > team_games['pts_allowed']).astype(int)

    # --- Días de Descanso ---
    team_games['rest_days'] = team_games.groupby('team_id')['game_date'].diff().dt.days
    team_games['rest_days'] = team_games['rest_days'].fillna(14).clip(upper=14)

    # --- Medias de Puntos y % de Victorias (Últimos 5 partidos) ---
    team_games['avg_pts_scored_5'] = team_games.groupby('team_id')['pts_scored'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    team_games['avg_pts_allowed_5'] = team_games.groupby('team_id')['pts_allowed'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    # NUEVO: % de victorias en los últimos 5 partidos (Racha anímica)
    team_games['win_pct_5'] = team_games.groupby('team_id')['won_game'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.5)

    home_features = team_games[team_games['is_home'] == 1][['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'win_pct_5']]
    home_features.columns = ['game_date', 'team_id_home', 'rest_days_home', 'avg_pts_scored_home', 'avg_pts_allowed_home', 'win_pct_5_home']

    away_features = team_games[team_games['is_home'] == 0][['game_date', 'team_id', 'rest_days', 'avg_pts_scored_5', 'avg_pts_allowed_5', 'win_pct_5']]
    away_features.columns = ['game_date', 'team_id_away', 'rest_days_away', 'avg_pts_scored_away', 'avg_pts_allowed_away', 'win_pct_5_away']

    # Merge de nuevo con el dataset principal
    df_copy = pd.merge(df_copy, home_features, on=['game_date', 'team_id_home'], how='left')
    df_copy = pd.merge(df_copy, away_features, on=['game_date', 'team_id_away'], how='left')
    df_copy = df_copy.drop_duplicates(subset=['team_id_home', 'team_id_away', 'game_date'])

    # NUEVO: Variables Diferenciales (Deltas)
    df_copy['elo_diff'] = df_copy['elo_h'] - df_copy['elo_a']
    df_copy['rest_diff'] = df_copy['rest_days_home'] - df_copy['rest_days_away']

    return df_copy

In [5]:
# 3. Escalado de datos
def escalar_datos(df, df_test, cols_no_escalables, cols_escalables):
    scaler = StandardScaler()
    scaler.fit(df[cols_escalables])

    df_escalado = scaler.transform(df[cols_escalables])
    df_test_escalado = scaler.transform(df_test[cols_escalables])
    
    data_entrada = np.hstack([np.array(df[cols_no_escalables]), df_escalado])
    data_entrada_test = np.hstack([np.array(df_test[cols_no_escalables]), df_test_escalado])
    
    return data_entrada, data_entrada_test
def preparar_datos_salida(df):
    return np.column_stack((df['pts_home'].values, df['pts_away'].values))

In [6]:
# ----------------- PROCESAMIENTO DE DATOS -----------------

# Partidos con ELO1 (El que dio mejores resultados)
print("Cargando datos y calculando features avanzadas...")
df_partidos_elo1 = pd.read_csv('csv_red/partidos_elo1.csv')

# Ejecutar las dos funciones clave antes de dividir en entrenamiento/test
df_partidos_elo1 = generar_features_avanzadas(df_partidos_elo1)
df_partidos_elo1, cols_equipos = preparar_datos_ohe(df_partidos_elo1, cols_equipos)

partidos_elo1 = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) <= 2017] 
partidos_elo1_test = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) > 2017] 

# Añadir cols_avanzadas al escalador para que normalice los días de descanso y los puntos
todas_cols_escalables = cols_fecha + cols_elo + cols_avanzadas
data_entrada_elo1, data_entrada_elo1_test = escalar_datos(partidos_elo1, partidos_elo1_test, cols_equipos, todas_cols_escalables)
data_salida_elo1, data_salida_elo1_test = preparar_datos_salida(partidos_elo1), preparar_datos_salida(partidos_elo1_test)

Cargando datos y calculando features avanzadas...


In [7]:
# ----------------- MODELO DE RED NEURONAL v1.3 -----------------
def crear_modelo_v1_3(n_input):
    modelo = tf.keras.Sequential([
        # Aumentamos ligeramente la primera capa para procesar las nuevas variables
        tf.keras.layers.Dense(256, activation='relu', input_shape=[n_input]),
        tf.keras.layers.Dropout(0.3), 
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2), 
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.1),
        tf.keras.layers.Dense(2, activation='linear') 
    ])
    
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        loss=tf.keras.losses.Huber(delta=1.5), 
        metrics=['mean_absolute_error']
    )
    return modelo

def evaluar_precision(modelo, entrada, salida, nombre_modelo):
    predicciones = modelo.predict(entrada, verbose=0)
    mae_home = mean_absolute_error(salida[:, 0], predicciones[:, 0])
    mae_away = mean_absolute_error(salida[:, 1], predicciones[:, 1])
    mae_total = mean_absolute_error(salida, predicciones)
    
    ganador_pred = (predicciones[:, 0] > predicciones[:, 1]).astype(int)
    ganador_real = (salida[:, 0] > salida[:, 1]).astype(int)
    precision = np.mean(ganador_pred == ganador_real) * 100

    print(f"\n--- Resultados {nombre_modelo} ---")
    print(f"Error Promedio Puntos (MAE Total): {mae_total:.2f}")
    print(f"  -> Error Medio Local: {mae_home:.2f}")
    print(f"  -> Error Medio Visitante: {mae_away:.2f}")
    print(f"Precisión Ganador (deducida): {precision:.2f}%")
    return precision, mae_total

In [8]:
# ----------------- ENTRENAMIENTO AVANZADO (Callbacks) -----------------
modelo_elo1_v1_3 = crear_modelo_v1_3(data_entrada_elo1.shape[1])
print("Entrenando Modelo v1_3 (Con Deltas, Rachas y Learning Rate Dinámico) ...")

# Optimizador de Learning Rate
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,       # Reduce a la mitad el learning rate
    patience=10,      # Si en 10 épocas no mejora
    min_lr=0.00001,   # Límite mínimo
    verbose=0
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=25, 
    restore_best_weights=True
)

history = modelo_elo1_v1_3.fit(
    data_entrada_elo1, data_salida_elo1, 
    epochs=2000, 
    verbose=0, 
    validation_split=0.1, 
    callbacks=[early_stopping, reduce_lr] # Aplica ambos callbacks
)
print("¡Modelo v1_3 entrenado!")

loss_v1_3 = modelo_elo1_v1_3.evaluate(data_entrada_elo1_test, data_salida_elo1_test, verbose=0)[0]
print(f"Pérdida (Huber Loss) Final: {loss_v1_3:.4f}")

acc_v1_3, mae_v1_3 = evaluar_precision(modelo_elo1_v1_3, data_entrada_elo1_test, data_salida_elo1_test, "Modelo ELO1_v1_3 Definitivo")

Entrenando Modelo v1_3 (Con Deltas, Rachas y Learning Rate Dinámico) ...
¡Modelo v1_3 entrenado!
Pérdida (Huber Loss) Final: 13.8976

--- Resultados Modelo ELO1_v1_3 Definitivo ---
Error Promedio Puntos (MAE Total): 9.99
  -> Error Medio Local: 10.05
  -> Error Medio Visitante: 9.94
Precisión Ganador (deducida): 64.33%


In [9]:
# ----------------- GUARDADO DE RESULTADOS -----------------
os.makedirs('resultados_finales', exist_ok=True)
def guardar_resultados_csv(df, modelo, entrada, nombre_archivo):
    df_copy = df.copy()
    df_copy = df_copy[['season_id', 'game_date', 'team_name_home', 'team_name_away', 'pts_home', 'pts_away']]
    predicciones = modelo.predict(entrada, verbose=0)
    df_copy['pred_pts_home'] = predicciones[:, 0].round(2)
    df_copy['pred_pts_away'] = predicciones[:, 1].round(2)
    df_copy['home_win'] = df_copy['pts_home'] > df_copy['pts_away']
    df_copy['pred_home_win'] = predicciones[:, 0] > predicciones[:, 1]
    df_copy['acierto'] = df_copy['home_win'] == df_copy['pred_home_win']
    df_copy.to_csv('resultados_finales/' + nombre_archivo, index=False)

guardar_resultados_csv(partidos_elo1_test, modelo_elo1_v1_3, data_entrada_elo1_test, 'resultados_modelo_v1_3.csv')
print("\nLos resultados se han guardado en 'resultados_finales/resultados_modelo_v1_3.csv'.")


Los resultados se han guardado en 'resultados_finales/resultados_modelo_v1_3.csv'.
